# 01b - Compressão JPEG para Remoção de Resíduo Estatístico

Imagens geradas por StyleGAN possuem assinaturas estatísticas específicas em alta frequência (artefatos de upsampling, padrões espectrais) que redes convolucionais aprendem trivialmente — sem precisar entender o conteúdo semântico da imagem.

Este notebook aplica compressão JPEG sobre todas as imagens do dataset para destruir esses artefatos, forçando o modelo a aprender características mais robustas.

O dataset comprimido é salvo em `data/processed/140k_faces_compressed/` e usado automaticamente pelos notebooks de treinamento.

In [ ]:
import io
from pathlib import Path

from PIL import Image
from tqdm import tqdm

In [ ]:
import shutil

PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"
SRC_DIR         = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
DST_DIR         = DATA_ROOT / "processed" / "140k_faces_compressed"
JPEG_QUALITY    = 50

splits  = ["train", "valid", "test"]
classes = ["real", "fake"]

# limpa o destino antes de recomprimir
if DST_DIR.exists():
    shutil.rmtree(DST_DIR)
    print("Pasta anterior removida:", DST_DIR)

for split in splits:
    for cls in classes:
        (DST_DIR / split / cls).mkdir(parents=True, exist_ok=True)

print("Fonte:", SRC_DIR)
print("Destino:", DST_DIR)
print("Qualidade JPEG:", JPEG_QUALITY)

## Compressão

In [ ]:
for split in splits:
    for cls in classes:
        src_folder = SRC_DIR / split / cls
        dst_folder = DST_DIR / split / cls
        images = sorted(src_folder.glob("*.jpg"))

        for img_path in tqdm(images, desc=f"{split}/{cls}"):
            dst_path = dst_folder / img_path.name
            img = Image.open(img_path).convert("RGB")
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=JPEG_QUALITY)
            buffer.seek(0)
            img_compressed = Image.open(buffer).copy()
            img_compressed.save(dst_path, format="JPEG", quality=JPEG_QUALITY)

import json
meta_path = DST_DIR / "metadata.json"
with open(meta_path, "w") as f:
    json.dump({"jpeg_quality": JPEG_QUALITY}, f)

print("Concluído.")
print("Metadata salvo em:", meta_path)

## Verificação

In [ ]:
import pandas as pd

rows = []
for split in splits:
    for cls in classes:
        count = len(list((DST_DIR / split / cls).glob("*.jpg")))
        rows.append({"split": split, "classe": cls, "imagens": count})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nTotal:", df["imagens"].sum())

## Comparação Visual: Original vs Comprimida

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
samples = list((SRC_DIR / "test" / "fake").glob("*.jpg"))[:4]

for col, img_path in enumerate(samples):
    orig = Image.open(img_path)
    comp = Image.open(DST_DIR / "test" / "fake" / img_path.name)

    axes[0, col].imshow(orig)
    axes[0, col].set_title("Original", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(comp)
    axes[1, col].set_title(f"JPEG q={JPEG_QUALITY}", fontsize=9)
    axes[1, col].axis("off")

plt.suptitle("Fake: Original vs Comprimida")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "figures" / "compressao_comparacao.png", dpi=150, bbox_inches="tight")
plt.show()